# ImmoEliza Project: Machine Learning model for price prediction

In this notebook, we will build a model to predict real estate prices.


### The plan:

**Data Cleaning**
- ✅ DROP columns:  
  - locality and region -> redundant with zip code
  - type of glazing (3 possibilities, annoying, captured by energy)
  - type of heating (8 possibilities, annoying to do, somewhat captured by energy, 8k+ nulls + 1700 not specified. almost half empty...) 
  ❓Might want to come back to this


- ✅ Categoricals
  - correlated trio (build_year, building_state, energy): keep all of them, let lasso pick the winner. 
  - Building state : keep it as one column. lasso will figure it out. One hot encode possible, but the weight should take care of it, and it's simpler with one column


- ✅ Property type / property subtype : We group then one hot encode towards this:
    prop_type_num = 
      flat + studentFlat + loft + flatStudio
      house + masterHouse + mansion
      villa
      mixedBuilding
      duple + triplex
      penthouse
      others = chalet, bungalow, cottage, groundFloor





- ✅ Fill in nulls:
  has_terrace : if null -> false
  Before or after the split makes no difference, it's a hard choice.

**SPLIT**
split into X_train, X_test
> don't forget to drop price in the test set

**Target Encoding for the zipcode**
My idea for now is to do this:
df.groupby(zipcode) get the median price per m2 for that zipcode and write that to a new column for each row. Basically, instead of a zip code i use the median price per m2 for that zipcode.
Reasonable?
> DROP zipcode after this
❗Carefull here and below. When doing this, only calculate values for the train set, and map them to the test set. Don't recalculate a new value for the test set, otherwise, i'm leaking info from the test set, into itself. instead, transform the test set with the values from the train set


**Filling Nulls**
building state (mode)

number of bedrooms (mode)
furnished (mode)
number of facade (mode)

build year, (median)
energy (median)
land area (median)

❗Carefull When doing this, only calculate values for the train set, and map them to the test set. Don't recalculate a new value for the test set, otherwise, i'm leaking info from the test set, into itself. instead, transform the test set with the values from the train set


**Polynomial Expansion**
Before standardisation because everything used needs to be normalised.


**Standardise**

**Model build**

Idea : loop through possible polynomial degrees. hyperparameter search
**Test and score the model**


In [ ]:
## Imports
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
plt.style.use('default') # seaborn, ggplot


# For the Model:
from sklearn.model_selection import train_test_split

from sklearn.base import BaseEstimator, TransformerMixin



# to redo the clean, start from clean_data_for_analysis.csv

# Cleaning done, working from model data
data_file = '../data/clean_data_for_model.csv'
df = pd.read_csv(data_file)


In [ ]:
## Data Cleaned : commented out so i can skip it 

""" 

### Dropping columns
df = df.drop(['locality', 'region', 'type_of_glazing', 'type_of_heating'], axis=1)

### resetting types
df['furnished'] = df['furnished'].astype('boolean')
df['has_terrace'] = df['has_terrace'].astype('boolean')
df['zip_code'] = df['zip_code'].astype('int')

## Fill in null in has terrace:
df["has_terrace"] = df["has_terrace"].fillna(False)

## Subtype Grouping : I drop property type and keep subtype, after grouping then like so: 
subtype_map = {
    'flat': 'flat',
    'studentFlat': 'flat',
    'loft': 'flat',
    'flatStudio': 'flat',
    'house': 'house',
    'masterHouse': 'house',
    'mansion': 'house',
    'villa': 'villa',
    'mixedBuilding': 'mixed_building',
    'duplex': 'duplex_triplex',
    'triplex': 'duplex_triplex',
    'penthouse': 'penthouse',
    'chalet': 'other',
    'bungalow': 'other',
    'cottage': 'other',
    'groundFloor': 'other',
}

df['prop_group'] = df['property_subtype'].map(subtype_map).fillna('other')
df = pd.get_dummies(df, columns=['prop_group'], drop_first=True)
df = df.drop(columns=['property_type', 'property_subtype'])


# Writing this to file, so it doesn't have to run everytime
df.to_csv("../data/clean_data_for_model.csv", index=False)  # saves without the row index


 """

In [ ]:
## Importing and Splitting data:

X = df.drop(columns=["y"]).to_numpy()  # We need to drop the target column
y = df.y.to_numpy().reshape(-1 , 1)    # here we do the reshaping in place.




In [2]:
## for the pipeline
from sklearn.base import BaseEstimator, TransformerMixin

class ZipCodeTargetEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # compute and store medians from training data
        self.medians_ = ...
        return self
    
    def transform(self, X):
        # map stored medians onto any data
        ...